# Fig. S1 — mechanical analogy for the surface profile

The radial equation $\partial_r^2\phi + \frac{2}{r}\partial_r\phi = f'(\phi)/\kappa$ read as
the motion of a unit mass in the inverted potential $-f(\phi)/\kappa$, with "time" $r$
and a drag $2/r$. One panel per concentration ($c/c_{\rm sat}$ = 0.2, 0.5, 0.8), all at
the parameters of Fig. 4D.

| element | meaning | source |
|---|---|---|
| curve | $-f(\phi)/\kappa$, $f(\phi)=f_b(\phi)-f_b(\phi_\infty)-f_b'(\phi_\infty)(\phi-\phi_\infty)$ | `nb`, `chi_b`, `kappa` of the Fig. 4D runs |
| balls | starting position $\phi(r{=}R)$ for $R$ = 20, 50 and 3000 nm | `phi_solver[k][0]` of `data/20261004/{small_R,medium_R,large_R}` |
| arrows | initial "speed" $\partial_r\phi\vert_R=-h_0/\kappa$: same for both $R$, drawn with a fixed length (not to scale) | — |
| dot + dotted line | final position $\phi(r\to\infty)=\phi_\infty$, the top of $-f$; enlarged in the inset | — |

**Data.** Same solver output as Fig. 4: run `bash figure_production/fig4/run_fig4_solver.sh`
from the repo root first if `data/20261004/` is missing. Built at `\textwidth`
(481.9 pt) for a `figure*`; style as in `FIGURE_STYLE.md`. Output: `figs1/figs1.pdf`.

In [ ]:
import json, os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import ConnectionPatch
from matplotlib.transforms import offset_copy

# Paths below are relative to the repo root (data/20261004/...), so chdir there.
# Where the kernel starts is not something to rely on, so look in three places.
REPO_OVERRIDE = None        # set to the repo path to skip the search entirely

SKIP = {"node_modules", "__pycache__", "venv", ".venv", "env", "build", "dist"}


def _is_repo(d):
    return (os.path.isfile(os.path.join(d, "Project.toml"))
            and os.path.isdir(os.path.join(d, "spec")))


def _up_from(d):
    d = os.path.abspath(d)
    for _ in range(8):
        if _is_repo(d):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return None
        d = parent
    return None


def _down_from(base, max_depth=4):
    level = [os.path.abspath(base)]
    for _ in range(max_depth):
        nxt = []
        for d in level:
            try:
                names = sorted(os.listdir(d))
            except OSError:
                continue
            for name in names:
                if name.startswith(".") or name in SKIP:
                    continue
                cand = os.path.join(d, name)
                if not os.path.isdir(cand) or os.path.islink(cand):
                    continue
                if _is_repo(cand):
                    return cand
                nxt.append(cand)
        level = nxt
    return None


def resolve_repo():
    if REPO_OVERRIDE:
        if not _is_repo(REPO_OVERRIDE):
            raise RuntimeError("REPO_OVERRIDE is not the repo root: " + REPO_OVERRIDE)
        return REPO_OVERRIDE
    nb = globals().get("__vsc_ipynb_file__") or globals().get("__file__")
    if nb:
        found = _up_from(os.path.dirname(os.path.abspath(nb)))
        if found:
            return found
    found = _up_from(os.getcwd()) or _down_from(os.getcwd())
    if found:
        return found
    raise RuntimeError("repo root not found from %s -- set REPO_OVERRIDE." % os.getcwd())


REPO = resolve_repo()
os.chdir(REPO)
print("working directory:", os.getcwd())

RUNS = {"small_R":  "data/20261004/small_R/demo_bvp_profiles.json",
        "medium_R": "data/20261004/medium_R/demo_bvp_profiles.json",
        "large_R":  "data/20261004/large_R/demo_bvp_profiles.json"}
for f in RUNS.values():
    print(("  ok      " if os.path.exists(f) else "  MISSING ") + f)

In [ ]:
def use_latex():
    plt.rcParams.update({"text.usetex": True, "font.family": "serif",
                         "text.latex.preamble": r"\usepackage{amsmath}"})
    try:
        fig = plt.figure()
        fig.text(0.5, 0.5, r"$-f(\phi)/\kappa$ $\phi_\infty$ $c/c_{\rm sat}$")
        fig.canvas.draw()
        plt.close(fig)
        return True
    except Exception as exc:
        plt.close("all")
        plt.rcParams.update({"text.usetex": False, "font.family": "serif",
                             "mathtext.fontset": "cm"})
        print("LaTeX unavailable, falling back to mathtext:",
              type(exc).__name__, str(exc).strip()[:120])
        return False


USETEX = use_latex()
print("usetex:", USETEX)

In [ ]:
# ---- model and solver output ------------------------------------------------
# Flory-Huggins per lattice site (kT = 1), as in fig4. f includes the chemical
# potential: f(phi) = fb(phi) - fb(phi_inf) - fb'(phi_inf)(phi - phi_inf), so its
# extremum at phi_inf is the top of the inverted potential -f/kappa (value 0).
D = {name: json.load(open(path)) for name, path in RUNS.items()}
p = D["large_R"]["params"]["surface"]
nb, chi, kappa, h0 = p["nb"], p["chi_b"], p["kappa"], p["omega_b"]
for d in D.values():           # the two runs differ only in R
    q = d["params"]["surface"]
    assert (q["nb"], q["chi_b"], q["kappa"], q["omega_b"]) == (nb, chi, kappa, h0)
    assert np.allclose(d["phi_inf"], D["large_R"]["phi_inf"])

NM_PER_SITE = 0.3


def fb(phi):
    return phi / nb * np.log(phi) + (1 - phi) * np.log(1 - phi) + chi * phi * (1 - phi)


def dfb(phi):
    return (np.log(phi) + 1) / nb - (np.log(1 - phi) + 1) + chi * (1 - 2 * phi)


def minus_f_over_kappa(phi, phi_inf):
    return -(fb(phi) - fb(phi_inf) - dfb(phi_inf) * (phi - phi_inf)) / kappa


phi_inf_grid = np.array(D["large_R"]["phi_inf"])
binodal = D["large_R"]["binodal"]
C_TARGETS = (0.2, 0.5, 0.8)                          # c/c_sat, one panel each
K = [int(np.argmin(np.abs(phi_inf_grid / binodal - c))) for c in C_TARGETS]

# phi at r = R: first point of the solver profile (z = 0 is the surface).
# The stored profile is the minimum-free-energy branch.
BALLS = {}
for name, d in D.items():
    BALLS[name] = [d["phi_solver"][k][0] for k in K]
    for k in K:
        assert d["success"][k], (name, k)
R_NM = {name: d["R"] * NM_PER_SITE for name, d in D.items()}

for c, k in zip(C_TARGETS, K):
    print("c/c_sat = %.4f  phi_inf = %.4g" % (phi_inf_grid[k] / binodal, phi_inf_grid[k]),
          "  ".join("phi(R=%.0f nm) = %.4f [%s]" % (R_NM[n], D[n]["phi_solver"][k][0],
                                                   D[n]["branch"][k]) for n in D))

In [ ]:
# ---- geometry and style (printed points; built at \textwidth) ---------------
PT_CAPTION, PT_TICK, PT_LETTER = 9, 8, 12
LW, LW_AUX, LW_AXES = 0.5, 0.4, 0.5

W = 481.9
LEFT, RIGHT, GAP = 46.0, 6.0, 10.0        # y label + ticks on A only; B, C share it
AXW = (W - LEFT - RIGHT - 2 * GAP) / 3
AXH = 96.0                                # Fig. 3 panel B's aspect (178 x 125 pt)
PAD_T, LETTER_H, XLAB, PAD_B = 1.0, 15.0, 26.0, 2.0
H = PAD_T + LETTER_H + AXH + XLAB + PAD_B
Y_AX = PAD_B + XLAB

RC = {
    "font.size": PT_CAPTION, "axes.labelsize": PT_CAPTION,
    "axes.titlesize": PT_CAPTION, "legend.fontsize": PT_CAPTION,
    "xtick.labelsize": PT_TICK, "ytick.labelsize": PT_TICK,
    "axes.spines.top": True, "axes.spines.right": True,
    "lines.linewidth": LW, "axes.linewidth": LW_AXES,
    "xtick.major.width": LW_AXES, "ytick.major.width": LW_AXES,
    "xtick.minor.width": LW_AUX, "ytick.minor.width": LW_AUX,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "xtick.major.pad": 1.5, "ytick.major.pad": 1.5,
    "axes.labelpad": 1.5, "axes.titlepad": 3.0,
}

# colours of R = 20, 50 and 3000 nm in Fig. 4D
COLORS = {"small_R": "#7FFF5F", "medium_R": "magenta", "large_R": "cyan"}
BALL_MS = 5.0                       # ball diameter, points
ARROW_LEN = 14.0                    # initial-speed arrow, points: fixed, NOT to scale
ARROW = dict(arrowstyle="-|>", mutation_scale=5, lw=LW, color="black",
             shrinkA=0, shrinkB=0)

Y_SCALE = 1e6                       # plot -f/kappa in units of 1e-6
XLIM = (0.0, 0.17)
YLIM = (-24.0, 3.0)                 # headroom: A's three-line legend clears the curve

# inset: the top of -f near phi_inf (axes fraction of each panel)
INSET = [0.10, 0.17, 0.38, 0.34]    # raised to leave room for its tick labels
INSET_XMAX = 3.0                    # inset spans phi in [0, INSET_XMAX * phi_inf]
INSET_EXP = -5                      # inset x ticks in units of 10^INSET_EXP, all panels


def _rect(x, yb, w, h):
    return [x / W, yb / H, w / W, h / H]


print("figure %.1f x %.1f pt; axes %.1f x %.1f pt" % (W, H, AXW, AXH))

In [ ]:
plt.rcParams.update(RC)
figs1 = plt.figure(figsize=(W / 72, H / 72))
phi = np.linspace(1e-9, XLIM[1], 4000)
AX = []

for j, (c, k) in enumerate(zip(C_TARGETS, K)):
    x0 = LEFT + j * (AXW + GAP)
    ax = figs1.add_axes(_rect(x0, Y_AX, AXW, AXH))
    AX.append(ax)
    q = phi_inf_grid[k]
    ax.plot(phi, minus_f_over_kappa(phi, q) * Y_SCALE, color="black", lw=LW, zorder=2)

    # final position: the top of the inverted potential, at phi_inf
    ax.plot([q, q], [YLIM[0], 0.0], ls=":", color="black", lw=LW, zorder=1)
    ax.plot([q], [0.0], "o", ms=2.5, mew=0, color="black", zorder=4, clip_on=False)

    # starting positions phi(R), with the initial speed pointing to smaller phi
    # arrows under the balls: a ball's arrow may run beneath its smaller-phi
    # neighbour (panel A, where the three start close together)
    for name in ("small_R", "medium_R", "large_R"):
        xb = BALLS[name][j]
        yb = minus_f_over_kappa(xb, q) * Y_SCALE
        ax.plot([xb], [yb], "o", ms=BALL_MS, mfc=COLORS[name], mec="black",
                mew=LW, zorder=6, label=r"$R = %.0f$ nm" % R_NM[name])
        ax.annotate("", xy=(xb, yb), xytext=(xb, yb),
                    xycoords=offset_copy(ax.transData, fig=figs1, x=-(BALL_MS / 2 + ARROW_LEN), units="points"),
                    textcoords=offset_copy(ax.transData, fig=figs1, x=-BALL_MS / 2, units="points"),
                    arrowprops=ARROW, zorder=5)

    ax.set_xlim(*XLIM); ax.set_ylim(*YLIM)
    ax.set_xticks([0.0, 0.05, 0.10, 0.15])
    ax.set_xlabel(r"Volume fraction $\phi$")
    ax.set_title(r"$c/c_{\rm sat} = %.1f$" % c)
    if j == 0:
        ax.set_ylabel(r"$-f(\phi)/\kappa$ $(\times 10^{-6})$")
    else:
        ax.tick_params(labelleft=False)

    # inset on the top of -f around phi_inf, linked to the dot in the main panel
    ins = ax.inset_axes(INSET)
    xi = np.linspace(1e-12, INSET_XMAX * q, 600)
    yi = minus_f_over_kappa(xi, q)
    ins.plot(xi, yi, color="black", lw=LW)
    ylo = yi.min()
    ins.set_xlim(0.0, INSET_XMAX * q)
    ins.set_ylim(ylo * 1.05, -0.25 * ylo)
    ins.plot([q, q], [ylo * 1.05, 0.0], ls=":", color="black", lw=LW)
    ins.plot([q], [0.0], "o", ms=2.5, mew=0, color="black", zorder=4)
    # x ticks: round multiples of 10^INSET_EXP, at most four
    span = INSET_XMAX * q / 10.0 ** INSET_EXP
    step = next(s for s in (1, 2, 5, 10, 20, 50) if span / s < 4)
    ticks = np.arange(0.0, span + 1e-9, step)
    ins.set_xticks(ticks * 10.0 ** INSET_EXP)
    ins.set_xticklabels(["%g" % t for t in ticks])
    ins.annotate(r"$\times 10^{%d}$" % INSET_EXP, xy=(1.0, 0.0), xycoords="axes fraction",
                 xytext=(2.0, -4.0), textcoords="offset points", ha="left", va="top",
                 fontsize=PT_TICK)
    ins.set_yticks([])
    ins.annotate(r"$\phi_\infty$", xy=(q, ylo * 1.05), xycoords="data",
                 xytext=(2.0, 2.0), textcoords="offset points", ha="left", va="bottom")
    # the magnified box in the main panel, and dotted leaders from the inset's
    # lower-left and upper-right corners to the matching corners of that box.
    # At the main panel's scale the box is far below a point in size (its width
    # 3 phi_inf ~ 1e-4, its height ~ 1e-9), so it sits on the phi_inf dot and
    # the two leaders converge there.
    box_lo = (0.0, ylo * 1.05 * Y_SCALE)
    box_hi = (INSET_XMAX * q, -0.25 * ylo * Y_SCALE)
    ax.add_patch(plt.Rectangle(box_lo, box_hi[0] - box_lo[0], box_hi[1] - box_lo[1],
                               fill=False, lw=LW_AUX, ec="black", clip_on=False))
    for xy_main, xy_ins in ((box_lo, (0.0, 0.0)), (box_hi, (1.0, 1.0))):
        figs1.add_artist(ConnectionPatch(xyA=xy_main, coordsA=ax.transData,
                                         xyB=xy_ins, coordsB=ins.transAxes,
                                         color="0.4", lw=LW_AUX, ls=":", zorder=1))

lg = AX[0].legend(loc="upper right", frameon=False, borderaxespad=0.3,
                  handletextpad=0.3, labelspacing=0.2, numpoints=1)

# name the arrow once, under the cyan (R = 3000 nm) arrow of panel B; right-aligned
# on the ball so the text sits in the empty space below the curve's dip
xb = BALLS["large_R"][1]
yb = minus_f_over_kappa(xb, phi_inf_grid[K[1]]) * Y_SCALE
AX[1].annotate("initial speed\n" + r"$-h_0/\kappa$", xy=(xb, yb), xycoords="data",
               xytext=(-BALL_MS / 2, -4.0), textcoords="offset points",
               ha="right", va="top", multialignment="center", linespacing=1.1)

In [ ]:
# panel letters (Arial 12 pt) on the line above each panel: A flush with the
# page edge, B and C level with the left edge of their axes
y_let = H - PAD_T
for j, letter in enumerate("ABC"):
    x_let = 1.0 if j == 0 else LEFT + j * (AXW + GAP) - 3.0
    figs1.text(x_let / W, y_let / H, letter, usetex=False, fontfamily="Arial",
               fontsize=PT_LETTER, ha="left", va="top")

OUT_DIR = os.path.join("figure_production", "figs1")
os.makedirs(OUT_DIR, exist_ok=True)
path = os.path.join(OUT_DIR, "figs1.pdf")
figs1.savefig(path, format="pdf")      # no bbox_inches: keeps \textwidth
print("Saved figure to", path)